# 03. Group-wise PCA with missingness masks

Adapted from Naomi's `groupwise_pca_with_mask.ipynb`.

Only change from the original: `cohort_group` is treated as an identifier
column, not a feature, and is carried through to the output for downstream
splitting.

For each original feature this notebook creates a mask (`1=observed`, `0=missing`).
Numeric values are standardized using observed values only, missing standardized
values are filled with zero, and PCA is fit **separately within each feature group**
across **all 39,716 patients combined** (so AD and non-AD embeddings live in the same
basis and are directly comparable). All group components are retained.

The x0 categorical demographics (gender, race, ethnicity) are one-hot encoded.
The final feature matrix concatenates one-hot demographics + group-wise PCA scores,
ordered x0 -> x42.

Outputs (in `output/pca/`):
- `X_features_pca.parquet`  (features, includes `cohort_group`)
- `X_mask.parquet`          (masks, separate, includes `cohort_group`)
- `Y_phecode.parquet`       (labels)
- `model_input_metadata.csv`


## 1. Setup and configuration

In [ ]:
# !pip install scikit-learn openpyxl pyarrow
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

INPUT_DIR = Path('output')
OUTPUT_DIR = INPUT_DIR / 'pca'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FEATURE_MATRIX_PATH = INPUT_DIR / '07_feature_matrix.parquet'
DISEASE_MATRIX_PATH = INPUT_DIR / '08_disease_matrix.parquet'
DISEASE_NAMES_PATH = INPUT_DIR / 'disease_names.csv'
GROUP_PATH_CANDIDATES = [INPUT_DIR / 'omop_price.xlsx', Path('omop_price.xlsx')]

ROW_ID = 'row_id'
PERSON_ID = 'person_id'
INDEX_TIME = 'index_datetime'
COHORT_GROUP_COL = 'cohort_group'   # identifier-like column added in notebook 02

DROP_PERSON_ID = False  # keep person_id for downstream cohort-level analysis
X0 = 'x0'
UNMAPPED_GROUP = 'x_unmapped'
PCA_DECIMALS = None

MISSING_CATEGORICAL_VALUES = [
    'I Do Not Know', 'I Prefer Not To Share', 'I Do Not See My Race Listed Here',
    'Unknown', 'X', 'Nonbinary (NB)'
]
MIN_REAL_CATEGORY_PERCENT = 0.5

print('Feature matrix:', FEATURE_MATRIX_PATH)
print('Disease matrix:', DISEASE_MATRIX_PATH)
print('Output:', OUTPUT_DIR.resolve())


## 2. Load and align X and Y

In [ ]:
if not FEATURE_MATRIX_PATH.exists():
    raise FileNotFoundError(FEATURE_MATRIX_PATH)
if not DISEASE_MATRIX_PATH.exists():
    raise FileNotFoundError(DISEASE_MATRIX_PATH)

X_raw = pd.read_parquet(FEATURE_MATRIX_PATH)
Y_raw = pd.read_parquet(DISEASE_MATRIX_PATH)

if ROW_ID not in X_raw or ROW_ID not in Y_raw:
    raise ValueError('row_id missing')
if X_raw[ROW_ID].duplicated().any() or Y_raw[ROW_ID].duplicated().any():
    raise ValueError('duplicate row_id')
if not X_raw[ROW_ID].reset_index(drop=True).equals(Y_raw[ROW_ID].reset_index(drop=True)):
    Y_raw = X_raw[[ROW_ID]].merge(Y_raw, on=ROW_ID, how='left')
if not X_raw[ROW_ID].reset_index(drop=True).equals(Y_raw[ROW_ID].reset_index(drop=True)):
    raise ValueError('X/Y row_id alignment failed')

print('X raw:', X_raw.shape)
print('Y raw:', Y_raw.shape)
print('Cohort distribution:')
print(X_raw[COHORT_GROUP_COL].value_counts())


## 3. Load x0-x42 group mapping from omop_price.xlsx

In [ ]:
group_path = next((p for p in GROUP_PATH_CANDIDATES if p.exists()), None)
if group_path is None:
    raise FileNotFoundError('Place omop_price.xlsx in the working dir or under output/.')

_xls = pd.ExcelFile(group_path)
_sheet = next(
    (s for s in _xls.sheet_names
     if {'xn', 'FullColumnName'}.issubset(pd.read_excel(group_path, sheet_name=s, nrows=0).columns)),
    _xls.sheet_names[0]
)
groups = pd.read_excel(group_path, sheet_name=_sheet)
print('Group sheet:', _sheet)
required = {'xn', 'FullColumnName'}
if not required.issubset(groups.columns):
    raise ValueError(f'Missing group columns: {required - set(groups.columns)}')

groups = groups.dropna(subset=['FullColumnName']).copy()
groups['xn'] = groups['xn'].astype(str)
groups['FullColumnName'] = groups['FullColumnName'].astype(str)
feature_to_group = dict(zip(groups['FullColumnName'], groups['xn']))

print('Mapping file:', group_path)
print('Mapped features:', len(feature_to_group))
display(groups[['xn', 'FullColumnName']].head())


## 4. Identify features and build missingness masks

In [ ]:
identifier_cols = {ROW_ID, PERSON_ID, INDEX_TIME, COHORT_GROUP_COL}
original_cols = [c for c in X_raw.columns if c not in identifier_cols]
numeric_cols = [c for c in original_cols if pd.api.types.is_numeric_dtype(X_raw[c])]
categorical_cols = [c for c in original_cols if c not in numeric_cols]
feature_group = {c: feature_to_group.get(c, UNMAPPED_GROUP) for c in original_cols}

unmapped = [c for c in original_cols if feature_group[c] == UNMAPPED_GROUP]
if unmapped:
    print(f'WARNING: {len(unmapped)} unmapped features assigned to {UNMAPPED_GROUP}')
    display(pd.DataFrame({'unmapped_feature': unmapped}))

_missing_set = {str(s).strip().lower() for s in MISSING_CATEGORICAL_VALUES} | {''}

def normalize_categorical(series):
    s = series.astype('string')
    is_missing = s.isna() | s.str.strip().str.lower().isin(_missing_set)
    return s.where(~is_missing, '__missing__'), is_missing

_mask_cols = {}
for c in original_cols:
    if c in categorical_cols:
        _, _is_missing = normalize_categorical(X_raw[c])
        _mask_cols[f'{c}__mask'] = (~_is_missing).astype('uint8')
    else:
        _mask_cols[f'{c}__mask'] = X_raw[c].notna().astype('uint8')
X_mask = pd.DataFrame(_mask_cols)

print('Original features:', len(original_cols))
print('Numeric:', len(numeric_cols), 'Categorical:', len(categorical_cols))
print('Mask shape:', X_mask.shape, 'Mask values:', sorted(np.unique(X_mask.to_numpy()).tolist()))
for c in categorical_cols:
    print(f'  {c}: {int((X_mask[c + "__mask"] == 0).sum())} missing (NaN/empty/non-response)')


## 5. Standardize observed numeric values, fill missing with zero

In [ ]:
zcols = {}
stats = []
for c in numeric_cols:
    v = pd.to_numeric(X_raw[c], errors='coerce')
    n = int(v.notna().sum())
    mean = float(v.mean()) if n else np.nan
    sd = float(v.std(ddof=0)) if n else np.nan
    if n == 0:
        z = pd.Series(np.nan, index=v.index, dtype='float64')
        note = 'all_missing'
    elif not np.isfinite(sd) or sd == 0:
        z = v - mean
        note = 'zero_sd_centered'
    else:
        z = (v - mean) / sd
        note = 'standardized_observed'
    zcols[c] = z.astype('float32')
    stats.append({
        'feature_name': c, 'xn': feature_group[c],
        'n_observed': n, 'n_missing': len(v) - n,
        'observed_fraction': n / len(v),
        'mean_observed': mean, 'sd_observed': sd, 'note': note,
    })

X_z = pd.DataFrame(zcols)
X_filled = X_z.fillna(0.0).astype('float32')
standardization_stats = pd.DataFrame(stats)
print('Standardized numeric shape:', X_z.shape)
print('Missing after zero fill:', int(X_filled.isna().sum().sum()))


## 6. One-hot encode x0 categorical demographics

In [ ]:
x0_categorical = [c for c in categorical_cols if feature_group[c] == X0]
demo_parts = []
metadata = []

if x0_categorical:
    print('COMBINED categorical proportions (missing -> __missing__; real <%.3g%% -> Other):'
          % MIN_REAL_CATEGORY_PERCENT)
    _norm_cat = {}
    for c in x0_categorical:
        _n = normalize_categorical(X_raw[c])[0]
        _vc = _n.value_counts(dropna=False)
        _total = int(_vc.sum())
        _rare = sorted([v for v, n in _vc.items()
                        if v != '__missing__' and (100.0 * n / _total) < MIN_REAL_CATEGORY_PERCENT])
        _coll = _n.where(~_n.isin(_rare), 'Other')
        _norm_cat[c] = _coll
        _fvc = _coll.value_counts(dropna=False); _M = int(_fvc.sum())
        print('')
        print(c + ':')
        for val, cnt in _fvc.items():
            print('   ' + str(int(cnt)).rjust(8) + ('  %6.2f%%  ' % (100 * cnt / _M)) + str(val))
        if _rare:
            print('   grouped into Other:', _rare)
    _norm_cat = pd.DataFrame(_norm_cat)
    cat = pd.get_dummies(_norm_cat, columns=x0_categorical, prefix=x0_categorical, dtype='int8')
    demo_parts.append(cat)
    for out in cat.columns:
        src_col = next(c for c in x0_categorical if out.startswith(c + '_'))
        metadata.append({
            'output_feature_name': out, 'output_type': 'x0_one_hot',
            'xn': X0, 'source_features': src_col,
        })

X_demo = pd.concat(demo_parts, axis=1) if demo_parts else pd.DataFrame(index=X_raw.index)
print('x0 one-hot output shape:', X_demo.shape)


## 7. PCA within each numeric feature group

In [ ]:
def group_key(x):
    m = re.fullmatch(r'x(\d+)', str(x))
    return (0, int(m.group(1))) if m else (1, str(x))

pca_groups = sorted({feature_group[c] for c in numeric_cols}, key=group_key)
score_parts = []
component_rows = []
group_rows = []
pc_source = {}

for group in pca_groups:
    cols = [c for c in numeric_cols if feature_group[c] == group]
    matrix = X_filled[cols].to_numpy(dtype=np.float32, copy=True)
    n_components = len(cols)
    pca = PCA(n_components=n_components, svd_solver='full')
    scores = pca.fit_transform(matrix).astype('float32')
    names = [f'{group}_PC{i:03d}' for i in range(1, n_components + 1)]
    score_parts.append(pd.DataFrame(scores, columns=names, index=X_raw.index))
    cumulative = np.cumsum(pca.explained_variance_ratio_)
    for i, name in enumerate(names):
        pc_source[name] = cols[i]
        component_rows.append({
            'output_feature_name': name, 'output_type': 'group_pca_score',
            'xn': group, 'source_features': ';'.join(cols),
            'component_number': i + 1,
            'explained_variance_ratio': float(pca.explained_variance_ratio_[i]),
            'cumulative_explained_variance_ratio': float(cumulative[i]),
        })
    group_rows.append({
        'xn': group, 'n_original_numeric_features': len(cols),
        'n_pca_components': n_components,
        'total_explained_variance_ratio': float(pca.explained_variance_ratio_.sum()),
        'original_features': ';'.join(cols),
    })
    print(f'{group}: {len(cols)} original features -> {n_components} PCs')

X_pca = pd.concat(score_parts, axis=1) if score_parts else pd.DataFrame(index=X_raw.index)

if X_pca.shape[1]:
    print('PCA score dtype :', X_pca.dtypes.iloc[0])
    print('Example raw values:', [float(v) for v in X_pca.iloc[0, :min(5, X_pca.shape[1])].tolist()])
    if PCA_DECIMALS is not None:
        X_pca = X_pca.round(PCA_DECIMALS).astype('float32')
        print(f'Rounded PCA scores to {PCA_DECIMALS} decimals.')

pca_group_summary = pd.DataFrame(group_rows)
metadata.extend(component_rows)
display(pca_group_summary)


## 8. Assemble final feature matrix + mask + Y

In [ ]:
# Keep ROW_ID, PERSON_ID, COHORT_GROUP_COL as identifiers in all outputs
id_cols = [ROW_ID]
if not DROP_PERSON_ID and PERSON_ID in X_raw:
    id_cols.append(PERSON_ID)
id_cols.append(COHORT_GROUP_COL)
ids = X_raw[id_cols].reset_index(drop=True)

def _gnum(x):
    s = str(x).replace('x', '')
    return int(s) if s.isdigit() else 10**9

_rank = {'x0_one_hot': 0, 'group_pca_score': 1}
_feat_meta = [m for m in metadata if m['output_type'] in _rank]
_order = sorted(range(len(_feat_meta)),
                key=lambda i: (_gnum(_feat_meta[i]['xn']),
                               _rank[_feat_meta[i]['output_type']], i))
ordered_feature_cols = [_feat_meta[i]['output_feature_name'] for i in _order]

_combined = pd.concat([X_demo.reset_index(drop=True),
                       X_pca.reset_index(drop=True)], axis=1)
if set(ordered_feature_cols) != set(_combined.columns):
    raise ValueError('feature column set mismatch between metadata and assembled matrix')
X_features = pd.concat([ids, _combined[ordered_feature_cols]], axis=1)

# Build 1:1 mask matrix
group_of = {m['output_feature_name']: m['xn'] for m in _feat_meta}
src_of = {}
for m in _feat_meta:
    if m['output_type'] == 'x0_one_hot':
        src_of[m['output_feature_name']] = m['source_features']
src_of.update(pc_source)

mask_data = {}
mask_meta = []
for col in ordered_feature_cols:
    src = src_of[col]
    mask_data[col] = X_mask[f'{src}__mask'].values
    mask_meta.append({
        'output_feature_name': col, 'output_type': 'missingness_mask',
        'xn': group_of[col], 'source_features': src,
    })
X_mask_out = pd.concat([ids, pd.DataFrame(mask_data, index=X_raw.index).reset_index(drop=True)],
                       axis=1)

# Y: also carry cohort_group through so downstream can split easily
Y_out = Y_raw.copy()
if PERSON_ID in Y_out and DROP_PERSON_ID:
    Y_out = Y_out.drop(columns=[PERSON_ID])
# Merge in cohort_group from X_raw
group_map = X_raw[[ROW_ID, COHORT_GROUP_COL]]
Y_out = Y_out.merge(group_map, on=ROW_ID, how='left')
# Reorder: row_id, cohort_group, ..., diseases
front = [ROW_ID, COHORT_GROUP_COL]
if PERSON_ID in Y_out:
    front.insert(1, PERSON_ID)
Y_out = Y_out[front + [c for c in Y_out.columns if c not in front]]

# Validate row alignment
for _name, _frame in [('features', X_features), ('mask', X_mask_out), ('Y', Y_out)]:
    if not _frame[ROW_ID].reset_index(drop=True).equals(ids[ROW_ID]):
        raise ValueError(f'{_name} row_id alignment failed')

model_input_metadata = pd.DataFrame(metadata + mask_meta)
print('X_features:', X_features.shape)
print('X_mask:', X_mask_out.shape)
print('Y:', Y_out.shape)


## 9. Save

In [ ]:
X_FEAT_PATH = OUTPUT_DIR / 'X_features_pca.parquet'
X_MASK_PATH = OUTPUT_DIR / 'X_mask.parquet'
Y_PATH = OUTPUT_DIR / 'Y_phecode.parquet'
META_PATH = OUTPUT_DIR / 'model_input_metadata.csv'
GROUP_SUM_PATH = OUTPUT_DIR / 'pca_group_summary.csv'
STATS_PATH = OUTPUT_DIR / 'standardization_stats.csv'
SUMMARY_PATH = OUTPUT_DIR / 'group_pca_summary.json'

X_features.to_parquet(X_FEAT_PATH, index=False)
X_mask_out.to_parquet(X_MASK_PATH, index=False)
Y_out.to_parquet(Y_PATH, index=False)
model_input_metadata.to_csv(META_PATH, index=False)
pca_group_summary.to_csv(GROUP_SUM_PATH, index=False)
standardization_stats.to_csv(STATS_PATH, index=False)

summary = {
    'n_patients': len(X_features),
    'pca_fit_population': 'all_patients_combined_AD_plus_non_AD',
    'pca_scope': 'within_groups',
    'missing_processing': 'standardize_observed_then_zero_fill',
    'feature_matrix_column_order': 'x0..x42 (one-hot demographics first within x0, then PCA scores)',
    'n_original_features': len(original_cols),
    'n_feature_columns': X_features.shape[1] - len(id_cols),
    'n_demographic_onehot_columns': X_demo.shape[1],
    'n_pca_scores': X_pca.shape[1],
    'n_disease_columns': Y_out.shape[1] - 2 - (1 if PERSON_ID in Y_out.columns else 0),
    'unmapped_features': unmapped,
    'cohort_counts': X_features[COHORT_GROUP_COL].value_counts().to_dict(),
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding='utf-8')

if DISEASE_NAMES_PATH.exists():
    pd.read_csv(DISEASE_NAMES_PATH).to_csv(OUTPUT_DIR / 'disease_names.csv', index=False)

for p in [X_FEAT_PATH, X_MASK_PATH, Y_PATH, META_PATH, GROUP_SUM_PATH, STATS_PATH, SUMMARY_PATH]:
    print('Saved:', p)
summary


## 10. Final validation

In [ ]:
print('Same rows (features vs Y):', len(X_features) == len(Y_out))
print('row_id order features vs Y:', X_features[ROW_ID].equals(Y_out[ROW_ID]))
print('row_id order mask vs Y    :', X_mask_out[ROW_ID].equals(Y_out[ROW_ID]))
print('Duplicate row_id:', bool(X_features[ROW_ID].duplicated().any()))
print('Every PCA group keeps all components:',
      bool((pca_group_summary.n_original_numeric_features == pca_group_summary.n_pca_components).all()))
print()
print('File sizes:')
for p in [X_FEAT_PATH, X_MASK_PATH, Y_PATH, META_PATH, GROUP_SUM_PATH, STATS_PATH, SUMMARY_PATH]:
    print(f'  {p.name:36s} {p.stat().st_size / 1024**2:10.2f} MB')
